In [9]:
%pip install nba_api

Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install --upgrade pandas


Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import os
from datetime import datetime, timedelta
from src.config import *
from src.utils import *
from src.nba_scraping import *
print(pd.__version__)

2.2.3


# -- Setup du run

In [12]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
season = ['2024-25'] # Mets la saison que tu veux
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# -- 1. Download nouveaux matchs (saison courante)

In [13]:
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, season[0])

last_games_path = download_games_for_seasons(season, matchs_output_dir, run_timestamp)

Extraction saison 2024-25
Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON'],
      dtype='object')


# -- 2. Comparaison avec l'historique pour extraire les nouveaux

In [14]:
hist_games_path = get_latest_file(DATA_GAMES_DIR)
games_to_scrape = get_new_games(hist_games_path, last_games_path)



#merge historical games with new games and remove duplicates
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})

print(historical_games.head(2))

all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

print(all_games.head(2))

save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'all_', run_timestamp)


6 nouveaux matchs à traiter
   SEASON_ID     TEAM_ID TEAM_ABBREVIATION        TEAM_NAME     GAME_ID  \
0      22000  1610612756               PHX     Phoenix Suns  0020000011   
1      22000  1610612765               DET  Detroit Pistons  0020000005   

    GAME_DATE    MATCHUP WL  MIN  PTS  ...  OREB  DREB  REB  AST  STL  BLK  \
0  2000-10-31  PHX @ GSW  L  240   94  ...    11    33   44   25   12    3   
1  2000-10-31  DET @ TOR  W  241  104  ...    11    34   45   21    7    6   

   TOV  PF  PLUS_MINUS   SEASON  
0   16  28        -2.0  2000-01  
1   12  27         9.0  2000-01  

[2 rows x 29 columns]
   SEASON_ID     TEAM_ID TEAM_ABBREVIATION        TEAM_NAME     GAME_ID  \
0      22000  1610612756               PHX     Phoenix Suns  0020000011   
1      22000  1610612765               DET  Detroit Pistons  0020000005   

    GAME_DATE    MATCHUP WL  MIN  PTS  ...  OREB  DREB  REB  AST  STL  BLK  \
0  2000-10-31  PHX @ GSW  L  240   94  ...    11    33   44   25   12    3   
1  2

'data/raw_last/games_merged/all__2025-05-23_18-22-39.csv'

# -- 3. Scrape boxscores pour ces matchs

In [15]:
scrape_boxscores_for_games(games_to_scrape, DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, batch_size=BATCH_SIZE)

[1/6] GAME_ID: 0042400311 - 2025-05-20
[2/6] GAME_ID: 0042400311 - 2025-05-20
[3/6] GAME_ID: 0042400301 - 2025-05-21
[4/6] GAME_ID: 0042400301 - 2025-05-21
[5/6] GAME_ID: 0042400312 - 2025-05-22
[6/6] GAME_ID: 0042400312 - 2025-05-22
✅ Batch 1 saved with 176 rows


True

# -- 4. Merge historique + nouveaux (batchs)

In [16]:
merge_boxscores_batches(
    hist_dir=DATA_BOXSCORES_BATCHES_MERGED_DIR,
    new_dir=os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp),
    out_dir=DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, 
    run_timestamp=run_timestamp
)

Merged boxscores from data/raw/boxscores/batches_merged/boxscores_full__2025-05-23_17-32-44.csv and data/raw_last/batches/2025-05-23_18-22-39  saved at data/raw_last/batches_merged/merged_boxscores_2025-05-23_18-22-39.csv


'data/raw_last/batches_merged/merged_boxscores_2025-05-23_18-22-39.csv'